<a href="https://colab.research.google.com/github/192521268simats-cell/machine-learning/blob/main/Experiment_3_%E2%80%94_Decision_Tree_using_ID3_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import math
from collections import Counter

# Training dataset
data = {
    "Outlook": ["Sunny", "Sunny", "Overcast", "Rain", "Rain",
                "Rain", "Overcast", "Sunny", "Sunny", "Rain",
                "Sunny", "Overcast", "Overcast", "Rain"],
    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool",
                    "Cool", "Cool", "Mild", "Cool", "Mild",
                    "Mild", "Mild", "Hot", "Mild"],
    "Humidity": ["High", "High", "High", "High", "Normal",
                 "Normal", "Normal", "High", "Normal", "Normal",
                 "Normal", "High", "Normal", "High"],
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Weak",
             "Strong", "Strong", "Weak", "Weak", "Weak",
             "Strong", "Strong", "Weak", "Strong"],
    "Play": ["No", "No", "Yes", "Yes", "Yes",
             "No", "Yes", "No", "Yes", "Yes",
             "Yes", "Yes", "Yes", "No"]
}

df = pd.DataFrame(data)
print("Training Dataset:")
print(df.to_string(index=False))

# Calculate entropy
def entropy(labels):
    counts = Counter(labels)
    total = len(labels)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

# Calculate information gain
def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])
    weighted_entropy = 0

    for value in df[attribute].unique():
        subset = df[df[attribute] == value]
        weighted_entropy += (
            len(subset) / len(df)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

# Build ID3 decision tree
def id3(df, attributes, target):
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    if not attributes:
        return df[target].mode()[0]

    gains = {
        attr: information_gain(df, attr, target)
        for attr in attributes
    }

    best = max(gains, key=gains.get)
    tree = {best: {}}

    for value in df[best].unique():
        subset = df[df[best] == value]

        if subset.empty:
            tree[best][value] = df[target].mode()[0]
        else:
            remaining = [a for a in attributes if a != best]
            tree[best][value] = id3(subset, remaining, target)

    return tree

# Display tree
tree = id3(df, list(df.columns[:-1]), "Play")

print("\nDecision Tree (ID3):")
print(tree)

# Classify a new sample
def predict(tree, sample, default="Yes"):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return default

    return predict(tree[attribute][value], sample, default)

new_sample = {
    "Outlook": "Sunny",
    "Temperature": "Cool",
    "Humidity": "Normal",
    "Wind": "Weak"
}

prediction = predict(tree, new_sample, df["Play"].mode()[0])

print("\nNew Sample:", new_sample)
print("Predicted Class:", prediction)


Training Dataset:
 Outlook Temperature Humidity   Wind Play
   Sunny         Hot     High   Weak   No
   Sunny         Hot     High Strong   No
Overcast         Hot     High   Weak  Yes
    Rain        Mild     High   Weak  Yes
    Rain        Cool   Normal   Weak  Yes
    Rain        Cool   Normal Strong   No
Overcast        Cool   Normal Strong  Yes
   Sunny        Mild     High   Weak   No
   Sunny        Cool   Normal   Weak  Yes
    Rain        Mild   Normal   Weak  Yes
   Sunny        Mild   Normal Strong  Yes
Overcast        Mild     High Strong  Yes
Overcast         Hot   Normal   Weak  Yes
    Rain        Mild     High Strong   No

Decision Tree (ID3):
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'Normal', 'Wind': 'Weak'}
Predicted Class: Yes
